# **Carga de los datos**

In [11]:
import os

base_dir = './cats_and_dogs_small'

train_dir = os.path.join(base_dir, 'train')
validation_dir = os.path.join(base_dir, 'validation')
test_dir = os.path.join(base_dir, 'test')

# Directorio con las imágenes de training
train_cats_dir = os.path.join(train_dir, 'cats')
train_dogs_dir = os.path.join(train_dir, 'dogs')

# Directorio con las imágenes de validación
validation_cats_dir = os.path.join(validation_dir, 'cats')
validation_dogs_dir = os.path.join(validation_dir, 'dogs')

# Directorio con las imágenes de test
test_cats_dir = os.path.join(test_dir, 'cats')
test_dogs_dir = os.path.join(test_dir, 'dogs')

# **Data Augmentation**

In [12]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

validation_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(train_dir,
                                                    batch_size=20,
                                                    class_mode='binary',
                                                    target_size=(150, 150))
validation_generator = validation_datagen.flow_from_directory(validation_dir,
                                                                batch_size=20,
                                                                class_mode='binary',
                                                                target_size=(150, 150))
test_generator = test_datagen.flow_from_directory(test_dir,
                                                    batch_size=20,
                                                    class_mode='binary',
                                                    target_size=(150, 150))

Found 2000 images belonging to 2 classes.
Found 1000 images belonging to 2 classes.
Found 1000 images belonging to 2 classes.


# **Obtención de una red ya entrenada**

Las redes se obtienen con el módulo *tensorflow.keras.applications*. Para este ejemplo de utilizará la red **VGG16** adaptandola al tamaño de nuestras imágenes de entrada.

In [1]:
from tensorflow.keras.applications import VGG16

pre_trained_model = VGG16(input_shape=(150, 150, 3), 
                                include_top=False, 
                                weights='imagenet')

2026-02-08 13:45:12.875429: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 8s 0us/step


In [2]:
pre_trained_model.summary()

Model: "vgg16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 150, 150, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 150, 150, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 75, 75, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 75, 75, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 75, 75, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 37, 37, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 37, 37, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 37, 37, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 37, 37, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 18, 18, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 18, 18, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 18, 18, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 18, 18, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 9, 9, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 9, 9, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 9, 9, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 9, 9, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 4, 4, 512)      │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,714,688 (56.13 MB)

 Trainable params: 14,714,688 (56.13 MB)

 Non-trainable params: 0 (0.00 B)

Ahora vamos a especificar que las capas convolucionales no deben ser entrenadas, lo que se denomina congelar, para ello, recorreremos todas las capas de la red y estableceremos su atributo *trainable* a **False**.

In [4]:
for layer in pre_trained_model.layers:
    layer.trainable = False

En Keras los modelos se considerán cómo capas, así que lo que tendríamos que hacer es crear un modelo aparte que es el que va a ser las capas densas cómo clasificador y luego entrenaríamos el modelo.

In [10]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Flatten, Dense

modelIFE = Sequential()

modelIFE.add(pre_trained_model)
modelIFE.add(Flatten())
modelIFE.add(Dense(256, activation='relu'))
modelIFE.add(Dense(1, activation='sigmoid'))

modelIFE.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 4, 4, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │     2,097,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,812,353 (64.13 MB)

 Trainable params: 2,097,665 (8.00 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

# **Entrenamiento de nuestra red resultante**

In [13]:
from tensorflow.keras.optimizers import RMSprop

modelIFE.compile(optimizer=RMSprop(learning_rate=1e-4),
              loss='binary_crossentropy',
              metrics=['acc'])

In [15]:
modelIFE.fit(train_generator, epochs=10, 
          validation_data=validation_generator, 
          steps_per_epoch=100, 
          validation_steps=50)

Epoch 1/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 112s 1s/step - acc: 0.8215 - loss: 0.3999 - val_acc: 0.8600 - val_loss: 0.3162
Epoch 2/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 103s 1s/step - acc: 0.8295 - loss: 0.3773 - val_acc: 0.8860 - val_loss: 0.2767
Epoch 3/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 112s 1s/step - acc: 0.8355 - loss: 0.3649 - val_acc: 0.8950 - val_loss: 0.2552
Epoch 4/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 101s 1s/step - acc: 0.8515 - loss: 0.3389 - val_acc: 0.8780 - val_loss: 0.2864
Epoch 5/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 110s 1s/step - acc: 0.8360 - loss: 0.3498 - val_acc: 0.8860 - val_loss: 0.2756
Epoch 6/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 106s 1s/step - acc: 0.8490 - loss: 0.3388 - val_acc: 0.8600 - val_loss: 0.3423
Epoch 7/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 117s 1s/step - acc: 0.8520 - loss: 0.3395 - val_acc: 0.9000 - val_loss: 0.2573
Epoch 8/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 178s 2s/step - acc: 0.8615 - loss: 0.3258 - val_acc: 0.8940 - val_loss: 0.2541
Epoch 9/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 99s 998m

In [17]:
test_loss, test_acc = modelIFE.evaluate(test_generator)
print('test acc:', test_acc)

50/50 ━━━━━━━━━━━━━━━━━━━━ 27s 547ms/step - acc: 0.8970 - loss: 0.2676
test acc: 0.8970000147819519
